# Solutions: Swapping embeddings

Reference solutions for `02_swapping-embeddings.ipynb`. Try the exercises yourself before reading these.

In [1]:
# Colab only
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/clariah2025-dse-ml/materials/TODO-path/

### Exercise 1

In [2]:
# 1. load letters_full.csv into df
import pandas as pd

df = pd.read_csv('../../../datasets/hsa/letters_full.csv')

# 2. find keywords that appear in 3+ languages

keyword_language = (
    df[["keywords", "language"]] # select the keyword column
    .dropna(subset=["keywords"]) # remove rows with missing keywords
    .assign(keyword=lambda d: d.keywords.str.split(";")) # split the keywords into lists
    .explode("keyword") # create a new row for each keyword
)

keyword_language["keyword"] = keyword_language["keyword"].str.strip() # get rid of leading/trailing whitespace
keyword_language = keyword_language[keyword_language.keyword != ""] # remove empty rows

keyword_language_counts = keyword_language.groupby("keyword")["language"].nunique() # group keywords by languages and check in how many languages each keyword appears
cross_lingual_keywords = set(keyword_language_counts[keyword_language_counts >= 3].index) # show which keywords appear in 3 or more different languages

print(len(cross_lingual_keywords), "keywords appear in 3+ languages") # show results




246 keywords appear in 3+ languages


In [3]:
# Extra: let's look at some texts with cross-lingual keywords

# choose a random cross-lingual keyword
import random
random_keyword = random.choice(list(cross_lingual_keywords))
print("Random cross-lingual keyword:", random_keyword)

# filter the df to show 3 rows with different languages containing the random_keyword
random_crosslingual_example = (
    df[df.keywords.str.contains(random_keyword, na=False)]
    .drop_duplicates(subset=["language"])
    .head(3)
)

# print out the texts for inspection
for i, row in random_crosslingual_example.iterrows():
    print(f"Language: {row.language}")
    print(f"Text: {row.text}")
    print("-" * 80)

Random cross-lingual keyword: Euskaltzaindia - Real Academia de la Lengua Vasca - Académie de la Langue Basque
Language: de
Text: Bereits 5 Tage nach Abgang meines letzten Briefes an Sie erhielt ich Ihre Antwort und bald darauf Ihre Druckschriften, für die ich Ihnen vielmals danke. Ich wollte Ihnen nun erst antworten, nachdem ich diese durchgearbeitet hatte, um Ihnen gleich meine Zweifel und Bemerkungen dazu mitzuteilen, doch da sie eine mühelose Fundgrube für meine Etymologien = Zusammenstellungen bilden und das Semester bald beginnt, wollte ich meine letzten Ferientage der Erledigung schwierigerer Aufgaben widmen, besonders dem Durchsuchen der letzten Bände der Revue de ling. der Werke Luchaires u. ä. nach brauchbarem Stoff. Nun erhalte ich gerade zum Feste als schönstes und willkommenstes Ostergeschenk Ihren zweiten Brief. Haben Sie für alles recht herzlichen Dank! Ich werde nun versuchen, Ihnen möglichst chronologisch, auf alles Erwünschte Antwort zu erteilen.
---------------------

In [4]:
# 3. filter df to rows tagged with at least one cross-lingual keyword

# we define a function that we can apply to each row of the df
def has_cross_lingual_keyword(keywords_field):
    if pd.isna(keywords_field): # check for missing keywords
        return False
    tags = {k.strip() for k in keywords_field.split(";")} # split the keywords into a set and remove leading/trailing whitespace
    return bool(tags & cross_lingual_keywords) # check if there is any overlap with the cross-lingual keywords


df_cross_lingual = df[df.keywords.apply(has_cross_lingual_keyword)] # filter the df to rows tagged with at least one cross-lingual keyword

# 4. sample ~333 rows from each of the 6 languages with enough data -> documents
balanced_languages = ["de", "fr", "it", "pt", "es", "en"]
per_language = 333

sample = pd.concat([
    df_cross_lingual[df_cross_lingual.language == lang].sample(n=per_language, random_state=42)
    for lang in balanced_languages
])

documents = sample.text.to_list()

# 5. print the language breakdown of the final sample
print(len(documents), "documents")

print(sample.language.value_counts())



1998 documents
language
de    333
fr    333
it    333
pt    333
es    333
en    333
Name: count, dtype: int64


In [5]:
# 6. build a combined multilingual stopword list with nltk

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

languages = ['german', 'french', 'italian', 'portuguese', 'spanish', 'english']
multilingual_stopwords = set()

for lang in languages:
    multilingual_stopwords.update(stopwords.words(lang))

# 7. create vectorizer_model with that stopword list

vectorizer_model = CountVectorizer(stop_words=list(multilingual_stopwords))


[nltk_data] Downloading package stopwords to C:\Users\alvares-
[nltk_data]     freire\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


`.explode("keyword")` turns each semicolon-separated `keywords` cell into one row per individual keyword, so `groupby("keyword")["language"].nunique()` can count how many distinct languages each keyword actually shows up in across the whole corpus, 246 keywords clear the 3-language bar. `has_cross_lingual_keyword` then keeps only the original rows (not the exploded ones) whose own keyword set overlaps with that list, about 20,000 of the ~42,000 rows qualify.

Sampling `per_language` documents separately from each of the six languages, rather than sampling 2000 rows from the filtered set all at once, is what actually guarantees the balance: German alone still makes up roughly two-thirds of even the filtered pool, so a single combined sample would still come out mostly German.

`vectorizer_model` gets built once here and reused, unchanged, in every `BERTopic(...)` below, the same way `verbose=True` is just a standing default rather than something to reconsider each time.

### Exercise 2

In [6]:
# create and fit a baseline topic_model_baseline, with vectorizer_model

from bertopic import BERTopic

topic_model_baseline = BERTopic(
    verbose=True, # enable verbose output     
    language="english", # set the language to English
    vectorizer_model=vectorizer_model # use the provided vectorizer model  
    )

topics, probs = topic_model_baseline.fit_transform(documents) # fit the model to the documents

# inspect get_topic_info()

topic_model_baseline.get_topic_info()

c:\Users\alvares-freire\Documents\Projects\dse-ml-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-22 11:37:21,460 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 63/63 [00:26<00:00,  2.39it/s]
2026-09-22 11:37:53,136 - BERTopic - Embedding - Completed ✓
2026-09-22 11:37:53,137 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-22 11:38:11,590 - BERTopic - Dimensionality - Completed ✓
2026-09-22 11:38:11,591 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-22 11:38:11,643 - BERTopic - Cluster - Completed ✓
2026-09-22 11:38:11,647 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-22 11:38:11,777 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,42,-1_ber_fr_spindel_verschiedene,"[ber, fr, spindel, verschiedene, berhaupt, ja,...","[Das mit dem ""weiten Überblick"", der den Roman..."
1,0,1066,0_pi_parte_ms_carta,"[pi, parte, ms, carta, tempo, qualche, ora, ex...",[Fin a tre anni fa io non conoscevo l'Ive che ...
2,1,320,1_basque_one_shall_think,"[basque, one, shall, think, would, may, much, ...",[You will readily understand that many of the ...
3,2,317,2_plus_jai_bien_tout,"[plus, jai, bien, tout, cette, faire, comme, c...",[Voilà pour le côté théorique de la question. ...
4,3,148,3_ber_fr_bitte_herr,"[ber, fr, bitte, herr, hofrat, ganz, schon, vi...",[Nach einem neuen Beschlusse der Akademie der ...
5,4,91,4_fr_ber_schon_ja,"[fr, ber, schon, ja, wrde, ganz, mehr, wohl, s...",[Ich bin mit dem Abschluß meiner „ Baskisch-ha...
6,5,14,5_schuchardt_prof_hugo_university,"[schuchardt, prof, hugo, university, chicago, ...","[Prof. H. Schuchardt –, Prof. H. Schuchardt., ..."


### Exercise 3

In [7]:
# compute the percentage of documents in topic -1 for the baseline model

info = topic_model_baseline.get_topic_info() # get topic information for the baseline model

outlier_pct_baseline = round(info.loc[info.Topic == -1, "Count"].sum() / info.Count.sum() * 100, 1) # compute the percentage of outliers for the baseline model

print(f"{outlier_pct_baseline}% outliers (baseline)")

2.1% outliers (baseline)


### Exercise 4

In [8]:
# create and fit topic_model_multilingual using the language argument and vectorizer_model
topic_model_multilingual = BERTopic(
    verbose=True,
    language="multilingual",
    vectorizer_model=vectorizer_model
    )
topics, probs = topic_model_multilingual.fit_transform(documents)

topic_model_multilingual.get_topic_info()

2026-09-22 11:38:11,914 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 63/63 [00:23<00:00,  2.68it/s]
2026-09-22 11:38:40,906 - BERTopic - Embedding - Completed ✓
2026-09-22 11:38:40,907 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-22 11:38:45,755 - BERTopic - Dimensionality - Completed ✓
2026-09-22 11:38:45,757 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-22 11:38:45,863 - BERTopic - Cluster - Completed ✓
2026-09-22 11:38:45,868 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-22 11:38:46,078 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,995,-1_basque_etc_plus_send,"[basque, etc, plus, send, bien, article, mr, o...",[so you disdain to give your reasons for not a...
1,0,123,0_basque_libro_article_trabajo,"[basque, libro, article, trabajo, edition, sen...",[to give me his edition of the O. M. of Pouvre...
2,1,104,1_portuguese_lingua_portuguez_português,"[portuguese, lingua, portuguez, português, one...",[Now it has happened to myself to visit old fr...
3,2,62,2_sempre_didl_scriva_felicitaciones,"[sempre, didl, scriva, felicitaciones, agradec...",[Só agora posso agradecer-lhe o seu apreciavel...
4,3,55,3_italia_roma_ver_firenze,"[italia, roma, ver, firenze, chiamato, lavorar...",[Finalmente dopo 8 giorni di peregrinazione ho...
5,4,46,4_basque_one_edition_basques,"[basque, one, edition, basques, paris, vinson,...",[probably start for Bayonne on the 8th of July...
6,5,45,5_pag_ecc_gergo_ie,"[pag, ecc, gergo, ie, dialetti, italiani, into...",[Denn in Halle hatt‘ ich ja gar nichts für Ihr...
7,6,42,6_lato_tonnara_barche_due,"[lato, tonnara, barche, due, rete, piccoli, ma...",[Detta rete calasi allo sgraro cioè non perpen...
8,7,42,7_tâ_diz_bâ_animar,"[tâ, diz, bâ, animar, cuisse, xinta, vadiar, p...","[l'ottima idea, mi pare, deve andare in fumo, ..."
9,8,41,8_artikel_vinson_still_may,"[artikel, vinson, still, may, pelle, smoke, ar...",[io ho messo tanto tempo a rispondere alla vs ...


### Exercise 5

In [9]:
# compute the percentage of documents in topic -1 for the multilingual model

info = topic_model_multilingual.get_topic_info()
outlier_pct_multilingual = round(info.loc[info.Topic == -1, "Count"].sum() / info.Count.sum() * 100, 1)

print(f"{outlier_pct_baseline}% outliers (baseline) vs {outlier_pct_multilingual}% (multilingual)")

# compare a few topics between the baseline and multilingual models
for topic_id in [0, 1, 2]:
    print("baseline:", topic_model_baseline.get_topic(topic_id))
    print("multilingual:", topic_model_multilingual.get_topic(topic_id))
    print()

2.1% outliers (baseline) vs 49.8% (multilingual)
baseline: [('pi', np.float64(0.015915084484343006)), ('parte', np.float64(0.013300518079597685)), ('ms', np.float64(0.012804366191296152)), ('carta', np.float64(0.012140689992815952)), ('tempo', np.float64(0.010905750512460021)), ('qualche', np.float64(0.010748481436217637)), ('ora', np.float64(0.01055305450101368)), ('exa', np.float64(0.009627704249485166)), ('revista', np.float64(0.009482326961719864)), ('amigo', np.float64(0.009135905472282513))]
multilingual: [('basque', np.float64(0.014419700409542094)), ('libro', np.float64(0.011347015583236352)), ('article', np.float64(0.010534013637044835)), ('trabajo', np.float64(0.010477277427162455)), ('edition', np.float64(0.010416637360033235)), ('sent', np.float64(0.010296792493981723)), ('copy', np.float64(0.009952671887365564)), ('say', np.float64(0.009879051415990452)), ('published', np.float64(0.009560693110757084)), ('revue', np.float64(0.009074873153260882))]

baseline: [('basque', np

Topic ids are assigned independently by each model, so "topic 0" in `topic_model_baseline` has no guaranteed relationship to "topic 0" in `topic_model_multilingual`, they're just both the largest non-outlier topic in their own run. Judge the two models by whether their word lists look coherent, not by matching ids, and with `vectorizer_model` applied from the start, that judgment is now actually meaningful: you're looking at real content words, not function words.

Because Exercise 1 deliberately sampled documents around keywords known to span several languages, this is a fairer test of the multilingual model than a blind random sample would be, but it's still worth checking your actual numbers rather than assuming the outcome. A plausible reading if the multilingual model still comes out worse here: `paraphrase-multilingual-MiniLM-L12-v2` is trained to pull semantically equivalent sentences from different languages close together, which suppresses language as a clustering signal, so it loses whatever "easy" signal the English-only model got from letters in the same language sitting close together by surface similarity, and has to rely entirely on the underlying cross-lingual thematic structure actually being strong enough for HDBSCAN to find dense regions in.

### Exercise 6

In [10]:
# 1 & 2. load the SentenceTransformer, create topic_model_manual with it and vectorizer_model

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
topic_model_manual = BERTopic(
    verbose=True, 
    embedding_model=embedding_model, 
    vectorizer_model=vectorizer_model
    )

# 3. fit it on documents, compare to topic_model_multilingual
topics, probs = topic_model_manual.fit_transform(documents)
print(len(topic_model_manual.get_topic_info()), "topics (manual)")
print(len(topic_model_multilingual.get_topic_info()), "topics (Exercise 4, shortcut)")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2679.65it/s]
2026-09-22 11:38:52,116 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 63/63 [00:23<00:00,  2.68it/s]
2026-09-22 11:39:15,633 - BERTopic - Embedding - Completed ✓
2026-09-22 11:39:15,634 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-22 11:39:19,296 - BERTopic - Dimensionality - Completed ✓
2026-09-22 11:39:19,297 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-22 11:39:19,369 - BERTopic - Cluster - Completed ✓
2026-09-22 11:39:19,372 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-22 11:39:19,493 - BERTopic - Representation - Completed ✓


3 topics (manual)
29 topics (Exercise 4, shortcut)


`language="multilingual"` sets `embedding_model` to this exact `SentenceTransformer` under the hood, so the two models are running the same embedding step (and both use the same `vectorizer_model`). Don't expect the topic counts to match exactly though: BERTopic's default UMAP step involves randomness, so two separate `fit_transform()` calls, even on identical input with the identical embedding model, will typically land on a similar but not identical number of topics.

### Exercise 7

In [11]:
# load paraphrase-multilingual-mpnet-base-v2, fit topic_model_larger with vectorizer_model

embedding_model_larger = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2") # load larger multilingual model

topic_model_larger = BERTopic(
    verbose=True, 
    embedding_model=embedding_model_larger, 
    vectorizer_model=vectorizer_model
)

topics, probs = topic_model_larger.fit_transform(documents) # fit

# compare outlier % AND number of topics found, against topic_model_baseline and topic_model_multilingual

for name, model in [        # iterate over all topic models to compare their performance
    ("baseline", topic_model_baseline),
    ("multilingual (MiniLM)", topic_model_multilingual),
    ("multilingual (mpnet)", topic_model_larger),
]:
    info = model.get_topic_info() # get topic information for the current model
    n_topics = (info.Topic != -1).sum() # count the number of non-outlier topics
    outlier_pct = round(info.loc[info.Topic == -1, "Count"].sum() / info.Count.sum() * 100, 1) # calculate the percentage of outlier documents
    print(f"{name}: {n_topics} topics, {outlier_pct}% outliers") 

for topic_id in [0, 1, 2]: # print the top 3 topics for each model
    print("multilingual (MiniLM):", topic_model_multilingual.get_topic(topic_id)) # print the top words for the current topic in the MiniLM model
    print("multilingual (mpnet):", topic_model_larger.get_topic(topic_id)) # print the top words for the current topic in the mpnet model
    print()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1562.27it/s]
2026-09-22 11:39:27,086 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 63/63 [01:19<00:00,  1.26s/it]
2026-09-22 11:40:46,246 - BERTopic - Embedding - Completed ✓
2026-09-22 11:40:46,247 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-22 11:40:50,559 - BERTopic - Dimensionality - Completed ✓
2026-09-22 11:40:50,561 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-22 11:40:50,627 - BERTopic - Cluster - Completed ✓
2026-09-22 11:40:50,630 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-22 11:40:50,810 - BERTopic - Representation - Completed ✓


baseline: 6 topics, 2.1% outliers
multilingual (MiniLM): 28 topics, 49.8% outliers
multilingual (mpnet): 36 topics, 52.4% outliers
multilingual (MiniLM): [('basque', np.float64(0.014419700409542094)), ('libro', np.float64(0.011347015583236352)), ('article', np.float64(0.010534013637044835)), ('trabajo', np.float64(0.010477277427162455)), ('edition', np.float64(0.010416637360033235)), ('sent', np.float64(0.010296792493981723)), ('copy', np.float64(0.009952671887365564)), ('say', np.float64(0.009879051415990452)), ('published', np.float64(0.009560693110757084)), ('revue', np.float64(0.009074873153260882))]
multilingual (mpnet): [('ora', np.float64(0.016618039647576462)), ('italia', np.float64(0.013364986212929903)), ('maurice', np.float64(0.01200688704410004)), ('roma', np.float64(0.01200688704410004)), ('réunion', np.float64(0.011694362936313665)), ('graz', np.float64(0.011157879612344218)), ('giorno', np.float64(0.010439402624454211)), ('avere', np.float64(0.010291617466371463)), ('mod

This is the actual payoff of `embedding_model=`: `paraphrase-multilingual-mpnet-base-v2` isn't reachable through `language=` at all, only by naming it directly.

Check both numbers before deciding mpnet won: a low outlier percentage on its own isn't success if it came from collapsing the corpus into one or two giant, undifferentiated topics rather than genuinely finding structure. If `get_topic(2)` (or higher ids) returns `False` for `topic_model_larger`, that's not an error, it's BERTopic's documented way of saying that topic id doesn't exist, which here would mean very few topics were actually found. Compare the topic *count* alongside the outlier percentage for all three models, and now that `vectorizer_model` has been applied from Exercise 2 onward, the word lists themselves are also a real, readable signal of whether mpnet's topics are genuinely coherent multilingual clusters or generic mega-topics.

**Don't stop at the outlier percentage, read the actual words.** One real run gave baseline 7 topics/2.2% outliers against mpnet's 32 topics/48.0%, which reads as a clear baseline win on the numbers alone. But baseline's biggest topic held over half the entire sample (1069 of 1998 documents) with words like `pi, parte, ms, carta, tempo, qualche, ora, exa, revista, amigo`, generic correspondence vocabulary (letter, time, journal, friend), not a specific subject. Its low outlier count partly reflects one giant catch-all bucket, not seven sharply differentiated themes.
```
[('pi', np.float64(0.01548181526167299)), ('parte', np.float64(0.012953914573335926)), ('ms', np.float64(0.012462552335200511)), ('carta', np.float64(0.011832666069142083)), ('tempo', np.float64(0.010636326410488417)), ('qualche', np.float64(0.010483883823988875)), ('ora', np.float64(0.010293267754461804)), ('exa', np.float64(0.00939689922091213)), ('revista', np.float64(0.009251450215483866)), ('amigo', np.float64(0.008919585818446488))]
```
A faster way to check for a specific theme than scrolling every topic: `topic_model_larger.find_topics("basque")` does a semantic search over the fitted topics and returns the closest matches by id, instead of eyeballing the whole table.